# 09 · Balanceo de Datos y Estrategias de Muestreo
## Módulo 3 — Feature Engineering & Modelado Supervisado

---

### 🎯 Objetivo del Cuaderno
Auditar la proporción de clases del Target (`Cluster`) en el conjunto de entrenamiento y evaluar empíricamente si la aplicación de técnicas de muestreo sintético o rebalanceo mejora el poder discriminante de los clasificadores.

### 🧪 Experimento Comparativo (Evaluado sobre el MISMO $X_{test}$ intacto):
Compararemos un modelo base (`RandomForestClassifier`) entrenado bajo 5 escenarios distintos:
1. **Sin Balanceo (Baseline):** Distribución original de entrenamiento.
2. **Random Over-Sampling (ROS):** Duplicación aleatoria de la clase minoritaria.
3. **Random Under-Sampling (RUS):** Submuestreo aleatorio de la clase mayoritaria.
4. **SMOTE (Synthetic Minority Over-sampling):** Generación de datos sintéticos basados en k-vecinos.
5. **Class Weight Balanced:** Ajuste de la función de pérdida sin alterar el dataset ($w_j = \frac{N}{2 \cdot n_j}$).

🔒 **Regla Metodológica:**
El balanceo se entrena e implementa **únicamente sobre $X_{train}$**. El conjunto $X_{test}$ permanece $100\%$ intacto con su distribución original de negocio.

In [1]:
# ==============================================================================
# BLOQUE 1: CARGA DE DATOS CON SELECCIÓN DE VARIABLES (MÓDULO 07)
# ==============================================================================
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Configuración visual
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# 1. Cargar bases reducidas a las 35 características óptimas (Módulo 07)
train_df = pd.read_csv('data_processed/07_train_sel.csv')
test_df = pd.read_csv('data_processed/07_test_sel.csv')

target_col = 'Cluster'

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

# 2. Diagnóstico del ratio de desbalance en Train y Test
tasa_train = y_train.mean() * 100
tasa_test = y_test.mean() * 100

print("="*75)
print("📊 DIAGNÓSTICO INICIAL DE BALANCE DE CLASES:")
print(f"· Train: {len(y_train)} registros | Clase 1: {y_train.sum()} ({tasa_train:.2f}%) | Clase 0: {(y_train==0).sum()} ({100-tasa_train:.2f}%)")
print(f"· Test:  {len(y_test)} registros | Clase 1: {y_test.sum()} ({tasa_test:.2f}%) | Clase 0: {(y_test==0).sum()} ({100-tasa_test:.2f}%)")
print("="*75)

📊 DIAGNÓSTICO INICIAL DE BALANCE DE CLASES:
· Train: 8018 registros | Clase 1: 4452 (55.53%) | Clase 0: 3566 (44.47%)
· Test:  2022 registros | Clase 1: 1124 (55.59%) | Clase 0: 898 (44.41%)


## 2. Experimento Comparativo de Balanceo sobre $X_{train}$

Entrenamos un clasificador estandarizado (`RandomForestClassifier`) sobre $X_{train}$ aplicando las distintas técnicas de muestreo, y medimos su capacidad discriminante real (**ROC-AUC, PR-AUC, Precision, Recall y F1-Score**) sobre el conjunto $X_{test}$ intacto.

In [4]:
# ==============================================================================
# BLOQUE 2: EJECUCIÓN DE TÉCNICAS DE BALANCEO Y BENCHMARK SOBRE TEST
# ==============================================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, average_precision_score
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
import joblib
import json
import os

# Definimos los balanceadores que alteran el dataset de Train
tecnicas_muestreo = {
    '1. Sin Balanceo (Baseline)': None,
    '2. Random Over-Sampling (ROS)': RandomOverSampler(random_state=42),
    '3. Random Under-Sampling (RUS)': RandomUnderSampler(random_state=42),
    '4. SMOTE': SMOTE(random_state=42),
    '5. Class Weight Balanced': 'class_weight'
}

resultados_balanceo = []
dict_datasets_train = {}

for nombre, sampler in tecnicas_muestreo.items():
    if nombre == '5. Class Weight Balanced':
        X_tr_bal, y_tr_bal = X_train.copy(), y_train.copy()
        model = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)
    elif sampler is None:
        X_tr_bal, y_tr_bal = X_train.copy(), y_train.copy()
        model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    else:
        X_tr_bal, y_tr_bal = sampler.fit_resample(X_train, y_train)
        model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
        
    # Guardamos el X_train balanceado para uso posterior si resultara útil
    dict_datasets_train[nombre] = (X_tr_bal, y_tr_bal)
    
    # Entrenamos el modelo
    model.fit(X_tr_bal, y_tr_bal)
    
    # Evaluamos en el MISMO Test Set sin tocar
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred_bin = (y_pred_proba >= 0.5).astype(int)
    
    auc_score = roc_auc_score(y_test, y_pred_proba)
    pr_auc = average_precision_score(y_test, y_pred_proba)
    prec = precision_score(y_test, y_pred_bin)
    rec = recall_score(y_test, y_pred_bin)
    f1 = f1_score(y_test, y_pred_bin)
    
    resultados_balanceo.append({
        'Estrategia': nombre,
        'Filas Train': len(X_tr_bal),
        '% Clase 1 Train': round(y_tr_bal.mean() * 100, 2),
        'ROC-AUC (Test)': round(auc_score, 4),
        'PR-AUC (Test)': round(pr_auc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4)
    })

df_res_balanceo = pd.DataFrame(resultados_balanceo)

print("🏆 TABLA COMPARATIVA DE ESTRATEGIAS DE BALANCEO (EVALUADAS EN TEST):")
display(df_res_balanceo)

# ------------------------------------------------------------------------------
# EXPORTACIÓN DEL DATASET SMOTE Y REPORTE JSON
# ------------------------------------------------------------------------------
os.makedirs('data_processed', exist_ok=True)
os.makedirs('reports', exist_ok=True)

# Guardamos el conjunto SMOTE generado (como requiere la arquitectura)
X_smote, y_smote = dict_datasets_train['4. SMOTE']
train_smote = X_smote.copy()
train_smote[target_col] = y_smote.values
train_smote.to_csv('data_processed/09_train_smote.csv', index=False)

reporte_bal = {
    'tasa_posicos_train_pct': round(float(y_train.mean()*100), 2),
    'tabla_comparativa': df_res_balanceo.to_dict(orient='records')
}

with open('reports/balanceo.json', 'w') as f:
    json.dump(reporte_bal, f, indent=4)

print("\n" + "="*75)
print("✅ ARTEFACTOS DE BALANCEO EXPORTADOS:")
print(f"· Train SMOTE guardado en: 'data_processed/09_train_smote.csv' ({train_smote.shape[0]} filas)")
print("· Reporte JSON guardado en: 'reports/balanceo.json'")
print("="*75)

🏆 TABLA COMPARATIVA DE ESTRATEGIAS DE BALANCEO (EVALUADAS EN TEST):


,Estrategia,Filas Train,% Clase 1 Train,ROC-AUC (Test),PR-AUC (Test),Precision,Recall,F1-Score
0,1. Sin Balanceo (Baseline),8018,55.5300,0.9880,0.9905,0.9317,0.9591,0.9452
1,2. Random Over-Sampling (ROS),8904,50.0000,0.9880,0.9904,0.9400,0.9475,0.9437
2,3. Random Under-Sampling (RUS),7132,50.0000,0.9878,0.9903,0.9486,0.9368,0.9427
3,4. SMOTE,8904,50.0000,0.9877,0.9902,0.9426,0.9502,0.9464
4,5. Class Weight Balanced,8018,55.5300,0.9880,0.9904,0.9449,0.9466,0.9458



✅ ARTEFACTOS DE BALANCEO EXPORTADOS:
· Train SMOTE guardado en: 'data_processed/09_train_smote.csv' (8904 filas)
· Reporte JSON guardado en: 'reports/balanceo.json'


## 3. Conclusiones del Experimento de Balanceo

La evaluación comparativa de técnicas de muestreo frente a la distribución real del conjunto de validación ($X_{test}$) arroja las siguientes conclusiones para el pase a producción:

---

### A. Diagnóstico de la Distribución Original
* **Ausencia de Desbalance Severo:** La variable Target (`Cluster`) presenta una distribución sumamente sana en el conjunto de entrenamiento, donde la Clase 1 (Motor Comercial) representa el **$55.53\%$** de la muestra.
* **No es un Evento Raro:** Al no enfrentarnos a incidencias menores al $10\%$ (como ocurre en detección de fraude o default de créditos raros), los clasificadores estándar no sufren del sesgo de ignorar a la clase minoritaria.

---

### B. Impacto Real de las Estrategias de Muestreo (Métricas en Test)
* **Estabilidad del ROC-AUC:** La capacidad de discriminación del modelo (ordenamiento de probabilidades) se mantuvo estática en **$\sim 0.9880$** a través de todas las técnicas (Baseline, SMOTE, ROS, RUS, Class Weight).
* **Ausencia de Ganancia Predictiva:** La inyección de datos sintéticos mediante **SMOTE** generó un ROC-AUC de $0.9877$ y un F1-Score de $0.9464$, variaciones microscópicas frente al Baseline original (F1 = $0.9452$). 

---

### C. Decisión Arquitectónica para Producción
* **Rechazo de Over/Under-Sampling Sintético:** Se dictamina **NO utilizar SMOTE ni ROS** en el pipeline final. Generar datos sintéticos aumentaría el costo computacional del entrenamiento sin aportar ninguna mejora real en la generalización.
* **Elección Óptima:** Para el entrenamiento final de los modelos, se utilizará el dataset con su distribución natural (Baseline) o, a lo sumo, el hiperparámetro `class_weight='balanced'` integrado de forma nativa en los algoritmos, garantizando un modelo ligero, rápido y calibrado.

---

### 🏁 Transición al Siguiente Módulo:
Habiendo depurado, seleccionado y validado la integridad y el balance de las características, procedemos al cuaderno **`10_consolidacion.ipynb`** (o equivalente de entrenamiento final), donde ensamblaremos los resultados y evaluaremos los modelos supervisados definitivos para su pase a producción.